# Notebook 03: Feature Engineering & Gold Layer

**Objective:** Create ML-ready features and Gold layer

**Gold Layer:** Feature-engineered data optimized for machine learning

---

## 1. Setup

In [ ]:
!pip install -q pyspark==3.4.0 pyarrow
print("✓ Libraries installed")

In [ ]:
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, hour, dayofweek, month as spark_month,
    sqrt, pow as spark_pow, lit, sin, cos, radians,
    year as spark_year
)
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

print("✓ Imports successful")

In [ ]:
spark = SparkSession.builder \
    .appName("NYC_Taxi_Feature_Engineering") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"✓ Spark {spark.version}")

## 2. Load Silver Layer

In [ ]:
silver_path = "data/silver/nyc_taxi_clean"
df_silver = spark.read.parquet(silver_path)

print(f"✓ Silver data loaded")
print(f"  Records: {df_silver.count():,}")
print(f"  Columns: {len(df_silver.columns)}")

## 3. Feature Engineering

### 3.1 Temporal Features

In [ ]:
# Extract temporal features
# Original implementation based on NYC Taxi domain knowledge

df_features = df_silver \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_day_of_week", dayofweek(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_month", spark_month(col("tpep_pickup_datetime")))

print("✓ Temporal features created")

### 3.2 Rush Hour Features

In [ ]:
# Define rush hours based on NYC traffic patterns
# Morning rush: 7-10 AM, Evening rush: 4-7 PM

df_features = df_features.withColumn(
    "is_rush_hour",
    when(
        ((col("pickup_hour") >= 7) & (col("pickup_hour") < 10)) |
        ((col("pickup_hour") >= 16) & (col("pickup_hour") < 19)),
        1
    ).otherwise(0)
)

print("✓ Rush hour feature created")

### 3.3 Weekend Feature

In [ ]:
# Weekend indicator (Saturday=7, Sunday=1 in Spark)
df_features = df_features.withColumn(
    "is_weekend",
    when((col("pickup_day_of_week") == 1) | (col("pickup_day_of_week") == 7), 1).otherwise(0)
)

print("✓ Weekend feature created")

### 3.4 Airport Trip Features

In [ ]:
# NYC Airport location IDs (from TLC data dictionary)
# JFK: 132, LaGuardia: 138, Newark: 1

airport_zones = [132, 138, 1]

df_features = df_features.withColumn(
    "is_airport_pickup",
    when(col("PULocationID").isin(airport_zones), 1).otherwise(0)
)

df_features = df_features.withColumn(
    "is_airport_dropoff",
    when(col("DOLocationID").isin(airport_zones), 1).otherwise(0)
)

df_features = df_features.withColumn(
    "is_airport_trip",
    when((col("is_airport_pickup") == 1) | (col("is_airport_dropoff") == 1), 1).otherwise(0)
)

print("✓ Airport features created")

### 3.5 Speed Feature

In [ ]:
# Calculate average speed (miles per hour)
df_features = df_features.withColumn(
    "avg_speed_mph",
    (col("trip_distance") / (col("trip_duration_minutes") / 60))
)

# Handle division by zero or extreme values
df_features = df_features.withColumn(
    "avg_speed_mph",
    when(col("avg_speed_mph").isNull(), 0)
    .when(col("avg_speed_mph") > 100, 100)
    .otherwise(col("avg_speed_mph"))
)

print("✓ Speed feature created")

### 3.6 Payment Type Features

In [ ]:
# Payment type indicators (1=Credit, 2=Cash, 3=No charge, 4=Dispute)
df_features = df_features.withColumn(
    "is_credit_card",
    when(col("payment_type") == 1, 1).otherwise(0)
)

df_features = df_features.withColumn(
    "is_cash",
    when(col("payment_type") == 2, 1).otherwise(0)
)

print("✓ Payment type features created")

### 3.7 Tip Percentage Feature

In [ ]:
# Calculate tip percentage
df_features = df_features.withColumn(
    "tip_percentage",
    when(col("fare_amount") > 0, (col("tip_amount") / col("fare_amount")) * 100).otherwise(0)
)

print("✓ Tip percentage feature created")

### 3.8 Feature Summary

In [ ]:
# Show sample with new features
df_features.select(
    "trip_distance",
    "fare_amount",
    "pickup_hour",
    "is_rush_hour",
    "is_weekend",
    "is_airport_trip",
    "avg_speed_mph",
    "tip_percentage"
).show(10)

In [ ]:
# Feature statistics
print("Feature Statistics:")
df_features.select(
    "is_rush_hour",
    "is_weekend",
    "is_airport_trip",
    "avg_speed_mph"
).summary().show()

## 4. Prepare ML Features

### 4.1 Select Feature Columns

In [ ]:
# Define feature columns for ML
feature_cols = [
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "is_rush_hour",
    "is_weekend",
    "is_airport_trip",
    "avg_speed_mph",
    "is_credit_card",
    "RatecodeID",
    "PULocationID",
    "DOLocationID"
]

# Target variable
target_col = "fare_amount"

print(f"Feature columns: {len(feature_cols)}")
print(f"Target: {target_col}")

### 4.2 Create Feature Vector

In [ ]:
# Assemble features into vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

df_assembled = assembler.transform(df_features)

print("✓ Features assembled into vector")

### 4.3 Scale Features

In [ ]:
# Standardize features (mean=0, std=1)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

print("✓ Features scaled")

### 4.4 Rename Target Column

In [ ]:
# Rename fare_amount to label for ML
df_gold = df_scaled.withColumnRenamed("fare_amount", "label")

print("✓ Target column renamed to 'label'")

## 5. Save Gold Layer

In [ ]:
# Select columns for Gold layer
gold_columns = [
    "features",
    "label",
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "is_rush_hour",
    "is_weekend",
    "is_airport_trip",
    "data_year",
    "data_month",
    "VendorID",
    "tpep_pickup_datetime"
]

df_gold_final = df_gold.select(gold_columns)

print(f"Gold layer columns: {len(gold_columns)}")

In [ ]:
gold_path = "data/gold/nyc_taxi_features"

print(f"Saving to {gold_path}...")

df_gold_final.write \
    .mode("overwrite") \
    .partitionBy("data_year", "data_month") \
    .parquet(gold_path)

print("✓ Gold layer saved")

## 6. Gold Layer Verification

In [ ]:
# Verify saved data
df_verify = spark.read.parquet(gold_path)

print("Gold Layer Verification:")
print(f"  Records: {df_verify.count():,}")
print(f"  Columns: {len(df_verify.columns)}")

In [ ]:
# Show schema
df_verify.printSchema()

In [ ]:
# Show sample
df_verify.select(
    "label",
    "trip_distance",
    "is_rush_hour",
    "is_weekend",
    "is_airport_trip"
).show(10)

In [ ]:
# Feature vector sample
print("Feature Vector Sample:")
df_verify.select("features").show(5, truncate=False)

## Summary

### Features Created

**Temporal Features:**
- ✓ Pickup hour, day of week, month
- ✓ Rush hour indicator (7-10 AM, 4-7 PM)
- ✓ Weekend indicator

**Location Features:**
- ✓ Airport trip indicators (JFK, LaGuardia, Newark)
- ✓ Pickup/dropoff location IDs

**Trip Features:**
- ✓ Average speed (mph)
- ✓ Trip distance
- ✓ Passenger count

**Payment Features:**
- ✓ Credit card indicator
- ✓ Tip percentage

### ML Preparation

- ✓ Feature vector assembled (13 features)
- ✓ Features standardized (mean=0, std=1)
- ✓ Target variable: fare_amount → label
- ✓ Gold layer saved as partitioned Parquet

### Next Steps

1. **Model Training:** Train 5 regression algorithms
2. **Hyperparameter Tuning:** Optimize model performance
3. **Evaluation:** Compare models using RMSE, R², MAE

---

**Academic Integrity:** All feature engineering is original implementation based on NYC Taxi domain knowledge and PySpark MLlib documentation.